# Cortex AI Gateway + LangChain Agent + MCP

This notebook demonstrates how to build a **LangChain agent** that:

1. Routes inference through **Snowflake's Cortex AI Gateway** (using a Snowflake-hosted Claude model)
2. Connects to **Snowflake data via MCP** (Model Context Protocol) for tool use
3. Produces **observability traces** you can query in `SNOWFLAKE.TELEMETRY.AGENT_TRACE_TABLE`

## Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│                     LangChain Agent (ReAct)                     │
│                                                                 │
│   LLM: ChatOpenAI ──► Cortex AI Gateway ──► Claude (Cortex)    │
│                                                                 │
│   Tools: MCP Client ──► Snowflake MCP Server                   │
│          ├── Cortex Analyst (SQL via semantic view)             │
│          └── Cortex Search (RAG over strategy docs)            │
└─────────────────────────────────────────────────────────────────┘
                              │
                              ▼
            SNOWFLAKE.TELEMETRY.AGENT_TRACE_TABLE
            SNOWFLAKE.ACCOUNT_USAGE.AI_GATEWAY_USAGE_HISTORY
```

## Prerequisites

1. Run `setup.sql` in your Snowflake account to create the database, tables, semantic view, search service, MCP server, and gateway route.
2. Create a [Personal Access Token (PAT)](https://docs.snowflake.com/en/user-guide/admin-pat) in Snowflake for API authentication.
3. Have your Snowflake account identifier (e.g., `myorg-myaccount`).

## 1. Install Dependencies

In [ ]:
%pip install langchain-openai langchain-mcp-adapters langgraph snowflake-connector-python

## 2. Configuration

The gateway and the MCP server both authenticate with a **programmatic access token (PAT)**.
Set two environment variables before starting Jupyter (or edit the defaults below):

```bash
export SNOWFLAKE_ACCOUNT=<org-account>          # e.g. myorg-myaccount
export SNOWFLAKE_USER=<user>
export SNOWFLAKE_PAT=<pat>                      # or SNOWFLAKE_PAT_FILE=/path/to/pat
```

Create a PAT with `ALTER USER <user> ADD PROGRAMMATIC ACCESS TOKEN ai_gateway_demo ROLE_RESTRICTION = 'SYSADMIN' DAYS_TO_EXPIRY = 7;`
(PATs are subject to your account's network and authentication policies.)


In [ ]:
import os
from pathlib import Path

SNOWFLAKE_ACCOUNT = os.environ["SNOWFLAKE_ACCOUNT"]
SNOWFLAKE_USER = os.environ["SNOWFLAKE_USER"]
SNOWFLAKE_PAT = os.environ.get("SNOWFLAKE_PAT") or Path(
    os.path.expanduser(os.environ.get("SNOWFLAKE_PAT_FILE", "~/.snowflake/ai_gateway_demo.pat"))
).read_text().strip()

# Hostnames need hyphens, not underscores, for a valid TLS certificate.
# Use the endpoint from SHOW AI GATEWAYS if your account uses private connectivity.
SF_HOST = f"{SNOWFLAKE_ACCOUNT}.snowflakecomputing.com".replace("_", "-").lower()
INFERENCE_BASE_URL = f"https://{SF_HOST}/api/v2/aigateways/SNOWFLAKE/v1"

MODEL = "openai-gpt-5.4"  # any model the gateway spec allows ('*' = all account models)

print(f"Account:       {SNOWFLAKE_ACCOUNT}")
print(f"User:          {SNOWFLAKE_USER}")
print(f"Inference URL: {INFERENCE_BASE_URL}")
print(f"Model:         {MODEL}")


## 3. Initialize the Gateway LLM

The Cortex AI Gateway exposes an **OpenAI-compatible** API. We use `ChatOpenAI` from LangChain,
pointing `base_url` at the inference endpoint and passing the PAT as the API key.

The `model` parameter is the actual model name (e.g., `claude-sonnet-4-6`). The gateway's
`models` allowlist controls which models are permitted — our spec uses `'*'` to allow all.

### Trace context propagation

For the gateway to group all inference calls from one agent turn under a single
`trace_id`, the client must send a [W3C `traceparent`](https://www.w3.org/TR/trace-context/) header on every request. Without it, each call gets its own trace ID and
they appear as unrelated events in `AGENT_TRACE_TABLE`.

We define a `make_llm()` factory that accepts an optional `traceparent` header.
In `run_agent()` (Section 5) we generate a fresh traceparent per invocation so
all LLM calls within one turn share the same trace.

In [ ]:
import uuid
from langchain_openai import ChatOpenAI


def make_traceparent() -> tuple[str, str]:
    """Generate a W3C traceparent header value and return (header_value, trace_id).

    Format: {version}-{trace_id}-{parent_id}-{flags}
    See https://www.w3.org/TR/trace-context/#traceparent-header
    """
    trace_id = uuid.uuid4().hex  # 32 hex chars
    parent_id = uuid.uuid4().hex[:16]  # 16 hex chars
    header = f"00-{trace_id}-{parent_id}-01"
    return header, trace_id


def make_llm(traceparent: str | None = None) -> ChatOpenAI:
    """Create a ChatOpenAI pointed at the Cortex AI Gateway.

    When *traceparent* is provided it is sent as a default header on every
    request so the gateway groups all inference calls under one trace_id.
    """
    headers = {}
    if traceparent:
        headers["traceparent"] = traceparent
    return ChatOpenAI(
        model=MODEL,
        base_url=INFERENCE_BASE_URL,
        api_key=SNOWFLAKE_PAT,
        temperature=0,
        max_completion_tokens=4096,  # GPT-5 family rejects max_tokens
        default_headers=headers or None,
    )


# Quick test - this goes through the gateway
llm = make_llm()
response = llm.invoke("What is ROAS in marketing? Answer in one sentence.")
print(response.content)

## 4. Connect MCP Tools

We use `langchain-mcp-adapters` to connect to the **Snowflake MCP server** object we created
in `setup.sql` (`MARKETING_MCP`). This MCP server exposes one tool:

- **`marketing_agent`** (Cortex Agent `MARKETING_AGENT`) — orchestrates Cortex Analyst, which generates and runs SQL against the semantic view, and Cortex Search for strategy documents

There is deliberately no raw SQL tool: exposed directly, Analyst only returns SQL text, and a
`SYSTEM_EXECUTE_SQL` tool would let the client run arbitrary SQL around the semantic view.

The MCP server endpoint follows the pattern:
```
https://<account>.snowflakecomputing.com/api/v2/databases/<db>/schemas/<schema>/mcp-servers/<name>/sse
```

`langchain-mcp-adapters` supports SSE transport directly — no local bridge or npx needed.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# MCP server endpoint — points to our MARKETING_MCP server object in Snowflake
# Snowflake hostnames require hyphens (not underscores) for valid SSL certs
MCP_HOST = f"{SNOWFLAKE_ACCOUNT}.snowflakecomputing.com".replace('_', '-')
MCP_ENDPOINT = (
    f"https://{MCP_HOST}"
    f"/api/v2/databases/CORTEX_GATEWAY_LAB/schemas/PUBLIC/mcp-servers/MARKETING_MCP"
)

# Streamable HTTP transport — connects directly to the Snowflake MCP server
mcp_config = {
    "snowflake": {
        "transport": "http",
        "url": MCP_ENDPOINT,
        "headers": {"Authorization": f"Bearer {SNOWFLAKE_PAT}"},
    }
}

print("MCP server configured (will connect when agent starts)")
print(f"  Transport: http")
print(f"  Endpoint:  {MCP_ENDPOINT}")

## 5. Build the LangChain Agent

We create a **ReAct agent** using LangGraph. The agent:
1. Receives a user question
2. Calls the `marketing_agent` MCP tool; the Cortex Agent behind it picks Cortex Analyst for data, Cortex Search for docs
3. Routes LLM inference through the Cortex AI Gateway
4. Synthesizes a final answer from the tool results

In [ ]:
from langgraph.prebuilt import create_react_agent

SYSTEM_PROMPT = """You are a marketing analytics assistant. For any question about campaign
performance data or marketing strategy, call marketing_agent with the user's question; it
queries governed Snowflake data and strategy documents and returns the answer.

Always cite specific numbers when available. Express ROI as a multiplier (e.g., 3.2x).
Round currency to 2 decimal places."""

# Initialize the MCP client and load tools once
mcp_client = MultiServerMCPClient(mcp_config)
tools = await mcp_client.get_tools()
print(f"Loaded {len(tools)} MCP tools: {[t.name for t in tools]}")


async def run_agent(question: str):
    """Run the agent against a question.

    Each invocation creates a fresh LLM with a unique W3C traceparent header
    so that every inference call the agent makes during this turn is grouped
    under one trace_id in AGENT_TRACE_TABLE.
    """
    traceparent, trace_id = make_traceparent()
    traced_llm = make_llm(traceparent=traceparent)
    print(f"Trace ID: {trace_id}")

    agent = create_react_agent(
        model=traced_llm,
        tools=tools,
        prompt=SYSTEM_PROMPT,
    )

    result = await agent.ainvoke({"messages": [{"role": "user", "content": question}]})

    # Print tool calls and responses for visibility
    for msg in result["messages"]:
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"Tool call: {tc['name']}({tc['args']})")
        if msg.type == 'tool':
            print(f"Tool response ({msg.name}): {msg.content[:500]}")
            print()

    final_message = result["messages"][-1]
    print("─" * 60)
    print(f"Q: {question}")
    print("─" * 60)
    print(final_message.content)
    return result

## 6. Run Queries

Let's ask the agent some marketing analytics questions. Each request flows through:

**User → LangChain Agent → AI Gateway (Claude) → MCP Tools (Snowflake) → Response**

All inference calls are logged by the gateway for observability.

In [ ]:
# Query 1: Structured data question (the agent behind MCP uses Cortex Analyst)
result1 = await run_agent("What was the ROAS by channel for Q4 2024?")

In [ ]:
# Query 2: Unstructured data question (the agent behind MCP uses Cortex Search)
result2 = await run_agent("What is our attribution methodology and how is revenue assigned to channels?")

In [ ]:
# Query 3: Hybrid question (may use both Analyst + Search)
result3 = await run_agent(
    "How did our Q4 Holiday Push campaign perform against the targets in the Q4 planning brief? "
    "Include actual ROAS and conversion numbers."
)

## 7. Observability — AI Gateway Traces

Every request through the AI Gateway is recorded with OpenTelemetry traces. These land in
the `AGENT_TRACE_TABLE('SNOWFLAKE')` table function, where each row is a **span** in a trace.

Key columns:
- `TRACE_ID` — groups all spans from one end-to-end request
- `SPAN_ID` — unique identifier for each operation
- `RECORD_ATTRIBUTES` — JSON with model name, token counts, etc.
- `TIMESTAMP` — when the span was recorded

Because we enabled `capture_payload.request_response: true` in the gateway spec,
request/response content is also captured in the trace attributes.

Let's query the observability data for the requests we just made.

In [ ]:
import snowflake.connector
import json

conn = snowflake.connector.connect(
    account=SNOWFLAKE_ACCOUNT,
    user=SNOWFLAKE_USER,
    authenticator="PROGRAMMATIC_ACCESS_TOKEN",
    token=SNOWFLAKE_PAT,
    warehouse="GATEWAY_LAB_WH",
)
cur = conn.cursor()
print(f"Connected as {SNOWFLAKE_USER} via PAT.")

In [ ]:
# Recent AI Gateway traces
cur.execute("""
    SELECT
        TRACE:trace_id::STRING AS TRACE_ID,
        RECORD:name::STRING AS SPAN_NAME,
        RECORD_ATTRIBUTES:"gen_ai.request.model"::STRING AS MODEL,
        RECORD_ATTRIBUTES:"gen_ai.usage.input_tokens"::INT AS IN_TOK,
        RECORD_ATTRIBUTES:"gen_ai.usage.output_tokens"::INT AS OUT_TOK,
        RECORD_ATTRIBUTES:"http.status_code"::INT AS HTTP
    FROM TABLE(AGENT_TRACE_TABLE('SNOWFLAKE'))
    WHERE TIMESTAMP > DATEADD('hour', -1, CURRENT_TIMESTAMP())
    ORDER BY TIMESTAMP DESC
    LIMIT 15
""")

rows = cur.fetchall()
print(f"Found {len(rows)} recent gateway trace spans\n")
print(f"{'TRACE_ID':>36} | {'SPAN_NAME':>25} | {'MODEL':>16} | {'IN':>6} | {'OUT':>6} | {'HTTP':>4}")
print("─" * 100)
for r in rows:
    print(f"{str(r[0]):>36} | {str(r[1]):>25} | {str(r[2]):>16} | {str(r[3]):>6} | {str(r[4]):>6} | {str(r[5]):>4}")

In [ ]:
# Credit usage summary from AI_GATEWAY_USAGE_HISTORY
# OPERATION_DETAILS is a JSON object keyed by model name with token counts
cur.execute("""
    SELECT
        f.key AS MODEL,
        COUNT(*) AS REQUEST_COUNT,
        SUM(f.value:"input_tokens"::INT) AS TOTAL_INPUT_TOKENS,
        SUM(f.value:"output_tokens"::INT) AS TOTAL_OUTPUT_TOKENS,
        SUM(CREDITS) AS TOTAL_CREDITS
    FROM SNOWFLAKE.ACCOUNT_USAGE.AI_GATEWAY_USAGE_HISTORY,
        LATERAL FLATTEN(input => OPERATION_DETAILS) f
    WHERE START_TIME > DATEADD('day', -1, CURRENT_TIMESTAMP())
    GROUP BY f.key
    ORDER BY TOTAL_CREDITS DESC
""")

rows = cur.fetchall()
print("AI Gateway Usage Summary (last 24h)\n")
print(f"{'MODEL':>20} | {'REQUESTS':>8} | {'IN_TOKENS':>10} | {'OUT_TOKENS':>10} | {'CREDITS':>10}")
print("─" * 70)
for row in rows:
    print(f"{str(row[0]):>20} | {row[1]:>8} | {row[2]:>10} | {row[3]:>10} | {row[4]:>10.6f}")

In [ ]:
# Detailed trace view — follow a single request end-to-end
cur.execute("""
    SELECT TRACE:trace_id::STRING AS TRACE_ID
    FROM TABLE(AGENT_TRACE_TABLE('SNOWFLAKE'))
    WHERE TIMESTAMP > DATEADD('hour', -1, CURRENT_TIMESTAMP())
    ORDER BY TIMESTAMP DESC
    LIMIT 1
""")
latest_trace = cur.fetchone()

if latest_trace:
    trace_id = latest_trace[0]
    print(f"Trace detail for: {trace_id}\n")

    cur.execute(f"""
        SELECT
            TRACE:span_id::STRING AS SPAN_ID,
            RECORD:name::STRING AS SPAN_NAME,
            TIMESTAMP,
            RECORD_ATTRIBUTES
        FROM TABLE(AGENT_TRACE_TABLE('SNOWFLAKE'))
        WHERE TRACE:trace_id::STRING = '{trace_id}'
        ORDER BY TIMESTAMP
    """)

    spans = cur.fetchall()
    for span in spans:
        attrs = json.loads(span[3]) if span[3] else {}
        print(f"  Span: {span[1]}")
        print(f"    Timestamp: {span[2]}")
        if "gen_ai.usage.input_tokens" in attrs:
            print(f"    Tokens: {attrs['gen_ai.usage.input_tokens']} in / {attrs['gen_ai.usage.output_tokens']} out")
        if "gen_ai.request.model" in attrs:
            print(f"    Model: {attrs['gen_ai.request.model']}")
        print()
else:
    print("No traces found yet — data may take a few minutes to appear.")

In [ ]:
# Deep dive: reconstruct the full agent conversation chain from a single trace
# This shows every reasoning step — tool calls, tool responses, and final answer.
cur.execute("""
    SELECT TRACE:trace_id::STRING AS TRACE_ID
    FROM TABLE(AGENT_TRACE_TABLE('SNOWFLAKE'))
    WHERE TIMESTAMP > DATEADD('hour', -2, CURRENT_TIMESTAMP())
      AND RECORD_ATTRIBUTES:"gen_ai.usage.input_tokens"::INT > 2000
    ORDER BY TIMESTAMP DESC
    LIMIT 1
""")
row = cur.fetchone()

if row:
    tid = row[0]
    cur.execute(f"""
        SELECT
            TRACE:span_id::STRING AS SPAN_ID,
            RECORD_ATTRIBUTES:"gen_ai.input.messages"::STRING AS INPUT_MSGS,
            RECORD_ATTRIBUTES:"gen_ai.output.messages"::STRING AS OUTPUT_MSGS,
            RECORD_ATTRIBUTES:"gen_ai.usage.input_tokens"::INT AS IN_TOK,
            RECORD_ATTRIBUTES:"gen_ai.usage.output_tokens"::INT AS OUT_TOK
        FROM TABLE(AGENT_TRACE_TABLE('SNOWFLAKE'))
        WHERE TRACE:trace_id::STRING = '{tid}'
        ORDER BY TIMESTAMP
    """)
    spans = cur.fetchall()
    print(f"Trace {tid} — {len(spans)} gateway call(s)\n")

    for i, span in enumerate(spans):
        msgs_in = json.loads(span[1]) if span[1] else []
        msgs_out = json.loads(span[2]) if span[2] else []
        print(f"═══ Gateway Call {i+1} ({span[3]} in / {span[4]} out tokens) ═══")

        for m in msgs_in:
            role = m.get('role', '?')
            parts = m.get('parts', [])
            for p in parts:
                if 'content' in p:
                    print(f"  [{role}] {p['content'][:200]}")
                elif 'name' in p:
                    print(f"  [{role}] → tool_call: {p['name']}({json.dumps(p.get('arguments',{}))[:150]})")
                elif 'response' in p:
                    resp_text = str(p['response'])[:200]
                    print(f"  [tool] ← {resp_text}")

        for m in msgs_out:
            parts = m.get('parts', [])
            for p in parts:
                if 'content' in p:
                    print(f"  [assistant] {p['content'][:300]}")
                elif 'name' in p:
                    print(f"  [assistant] → tool_call: {p['name']}({json.dumps(p.get('arguments',{}))[:150]})")
        print()
else:
    print("No multi-turn traces found in last 2 hours.")

## 8. Cost Management

The gateway is a natural unit for cost reporting: every inference request is attributed to the
Snowflake user who made it, with token counts and credit consumption recorded automatically.
This section covers how to monitor what the gateway is spending and how to limit it.

Snowflake offers two spending controls for gateway spend:

| Control | Scope | Enforcement | Use it when |
|---|---|---|---|
| **Shared resource budget** | A group of users identified by a tag, pooled | Notification, plus stored procedures you write | Several teams share the gateway and each needs its own limit |
| **Per-user quota** | Each user individually (limits are never pooled) | Built-in blocking, applied within minutes | You need usage to actually stop, not just alert |

Budget evaluation is periodic (up to 6.5 hours, or one hour with the low-latency option).
Quota blocks apply within minutes.

In [ ]:
# Gateway spend by user over the last 7 days
cur.execute("""
    SELECT
        u.NAME AS USER_NAME,
        f.key AS MODEL,
        COUNT(*) AS REQUEST_COUNT,
        SUM(f.value:"input_tokens"::INT) AS TOTAL_INPUT_TOKENS,
        SUM(f.value:"output_tokens"::INT) AS TOTAL_OUTPUT_TOKENS,
        SUM(g.CREDITS) AS TOTAL_CREDITS
    FROM SNOWFLAKE.ACCOUNT_USAGE.AI_GATEWAY_USAGE_HISTORY g
        JOIN SNOWFLAKE.ACCOUNT_USAGE.USERS u ON g.USER_ID = u.USER_ID
        ,LATERAL FLATTEN(input => g.OPERATION_DETAILS) f
    WHERE g.START_TIME > DATEADD('day', -7, CURRENT_TIMESTAMP())
    GROUP BY u.NAME, f.key
    ORDER BY TOTAL_CREDITS DESC
""")

rows = cur.fetchall()
print("AI Gateway Spend by User (last 7 days)\n")
print(f"{'USER':>20} | {'MODEL':>20} | {'REQUESTS':>8} | {'IN_TOKENS':>10} | {'OUT_TOKENS':>10} | {'CREDITS':>10}")
print("─" * 90)
for row in rows:
    print(f"{str(row[0]):>20} | {str(row[1]):>20} | {row[2]:>8} | {row[3]:>10} | {row[4]:>10} | {row[5]:>10.6f}")

### Shared resource budget

A shared resource budget lets you track gateway spend by team or cost center. The workflow is:

1. **Tag users** with a cost-center or team tag
2. **Create a budget** and add the tag so it tracks only those users
3. **Add the AI Gateway** as a shared resource on the budget
4. **Set a spending limit** — notifications fire when spend approaches it

When a tagged user makes a gateway request, the credits count toward that budget's spending limit.
You can configure notifications and custom actions (stored procedures) that fire at configurable thresholds.

In [ ]:
# --- Shared resource budget for gateway spend ---

# 1. Create a tag for cost-center attribution
cur.execute("""CREATE TAG IF NOT EXISTS CORTEX_GATEWAY_LAB.PUBLIC.COST_CENTER
               COMMENT = 'Team or cost center for gateway budget attribution'""")
print("1. Tag created:", cur.fetchone()[0])

# 2. Tag the current user
cur.execute(f"""ALTER USER "{SNOWFLAKE_USER.upper()}" SET TAG CORTEX_GATEWAY_LAB.PUBLIC.COST_CENTER = 'ENGINEERING'""")
print("2. User tagged:", cur.fetchone()[0])

# 3. Create the budget
cur.execute("""CREATE SNOWFLAKE.CORE.BUDGET IF NOT EXISTS CORTEX_GATEWAY_LAB.PUBLIC.GATEWAY_BUDGET()""")
print("3. Budget created:", cur.fetchone()[0])

# 4. Associate the user tag with the budget
cur.execute("""CALL CORTEX_GATEWAY_LAB.PUBLIC.GATEWAY_BUDGET!SET_USER_TAGS(
    [[(SELECT SYSTEM$REFERENCE('TAG', 'CORTEX_GATEWAY_LAB.PUBLIC.COST_CENTER', 'SESSION', 'APPLYBUDGET')),
      'ENGINEERING']],
    'UNION')""")
print("4. User tag set:", cur.fetchone()[0])

# 5. Add AI Gateway as a shared resource
cur.execute("""CALL CORTEX_GATEWAY_LAB.PUBLIC.GATEWAY_BUDGET!ADD_SHARED_RESOURCE('AI GATEWAY')""")
print("5. Shared resource added:", cur.fetchone()[0])

# 6. Set a 500-credit monthly spending limit
cur.execute("""CALL CORTEX_GATEWAY_LAB.PUBLIC.GATEWAY_BUDGET!SET_SPENDING_LIMIT(500)""")
print("6. Spending limit set:", cur.fetchone()[0])

# Verify the budget scope
cur.execute("""CALL CORTEX_GATEWAY_LAB.PUBLIC.GATEWAY_BUDGET!GET_BUDGET_SCOPE()""")
scope = cur.fetchone()[0]
print("\nBudget scope:")
print(json.dumps(json.loads(scope), indent=2))

### Per-user quota

A per-user quota enforces hard spending limits at the individual user level. Unlike budgets,
quotas can **block** users automatically when they reach their limit — no custom stored procedure
needed. Blocks are released when the cycle resets (monthly, weekly, or daily).

Per-user quotas support monthly, weekly, and daily limits evaluated independently. A user is
blocked as soon as they hit any one of them. Block enforcement is evaluated within minutes of
a spend event.

In [ ]:
# --- Per-user quota for gateway spend ---

# 1. Create the quota object
cur.execute("""CREATE SNOWFLAKE.CORE.QUOTA IF NOT EXISTS CORTEX_GATEWAY_LAB.PUBLIC.GATEWAY_QUOTA()""")
print("1. Quota created:", cur.fetchone()[0])

# 2. Add AI Gateway as the monitored domain
cur.execute("""CALL CORTEX_GATEWAY_LAB.PUBLIC.GATEWAY_QUOTA!ADD_SHARED_RESOURCE('AI GATEWAY')""")
print("2. Shared resource added:", cur.fetchone()[0])

# 3. Set per-user spending limits at each cycle
cur.execute("""CALL CORTEX_GATEWAY_LAB.PUBLIC.GATEWAY_QUOTA!SET_PER_USER_LIMIT(100)""")           # monthly
print("3a. Monthly limit:", cur.fetchone()[0])

cur.execute("""CALL CORTEX_GATEWAY_LAB.PUBLIC.GATEWAY_QUOTA!SET_PER_USER_LIMIT(30, 'WEEKLY')""")  # weekly
print("3b. Weekly limit:", cur.fetchone()[0])

cur.execute("""CALL CORTEX_GATEWAY_LAB.PUBLIC.GATEWAY_QUOTA!SET_PER_USER_LIMIT(10, 'DAILY')""")   # daily
print("3c. Daily limit:", cur.fetchone()[0])

# 4. Enable block enforcement — users are blocked when they hit any limit
cur.execute("""CALL CORTEX_GATEWAY_LAB.PUBLIC.GATEWAY_QUOTA!SET_BLOCK_ENFORCEMENT_ENABLED(TRUE, FALSE)""")
print("4. Block enforcement:", cur.fetchone()[0])

# Verify the quota config
cur.execute("""CALL CORTEX_GATEWAY_LAB.PUBLIC.GATEWAY_QUOTA!GET_CONFIG()""")
row = cur.fetchone()
cols = [desc[0] for desc in cur.description]
print("\nQuota config:")
for col, val in zip(cols, row):
    print(f"  {col}: {val}")

## Summary

This example demonstrated:

| Component | Role |
|---|---|
| **Cortex AI Gateway** | Centralized LLM routing — one endpoint, configurable routes, built-in observability |
| **LangChain + LangGraph** | Agent framework with ReAct reasoning over MCP tools |
| **MCP (Model Context Protocol)** | Standardized tool interface connecting the agent to Snowflake data services |
| **Cortex Analyst** | Natural language → SQL against the semantic view |
| **Cortex Search** | RAG retrieval over strategy documents |
| **AGENT_TRACE_TABLE** | OpenTelemetry traces for every gateway inference call |
| **AI_GATEWAY_USAGE_HISTORY** | Token usage and credit consumption per model/route |
| **Budgets & Per-user Quotas** | Spending controls — team-level budgets with notifications, or per-user hard blocks |

### Key Takeaways

1. **The gateway is OpenAI-compatible** — any SDK or framework that speaks the OpenAI chat completions API works by pointing `base_url` at `/api/v2/cortex/v1`.
2. **Model allowlists control access** — the `models` spec restricts which models can be called; use `'*'` for unrestricted or `'claude-*'` to lock down to specific families.
3. **MCP provides a clean tool boundary** — the agent doesn't need Snowflake credentials or SQL knowledge; MCP handles that.
4. **Observability is automatic** — every inference call is traced in `AGENT_TRACE_TABLE('SNOWFLAKE')` with token counts, model metadata, and optional payload capture.
5. **Cost controls are built in** — shared resource budgets track team-level spend with notifications; per-user quotas enforce hard blocks within minutes when a user hits their credit limit.